## laoding sql function 

In [0]:
from pyspark.sql import functions as F


In [0]:
%sql
use catalog centuryhealthdatabase;

In [0]:
## Helper Function : Clean Column Names
def clean_cols(df):
    for c in df.columns:
        new_col = c.strip().lower().replace(" ", "_")
        df = df.withColumnRenamed(c, new_col)
    return df

# Helper Function : Safe Date Conversion
# Invalid values like 9999-99-99 => NULL
def safe_date(col_name):
    return F.expr(f"try_cast({col_name} as date)")
    
# Helper Function : Safe Timestamp Conversion
def safe_timestamp(col_name):
    return F.expr(f"try_cast({col_name} as timestamp)")


## transforming  patients

In [0]:
patients = spark.table("bronze.patients")
patients = clean_cols(patients)

patients = patients \
.withColumn("birthdate", safe_date("birthdate")) \
.withColumn("deathdate", safe_date("deathdate")) \
.withColumn("gender", F.lower(F.trim(F.col("gender")))) \
.withColumn("marital", F.upper(F.trim(F.col("marital")))) \
.fillna({
    "gender": "unknown",
    "marital": "unknown"
}) \
.dropDuplicates()

# Fix Latitude Scaling Issue
patients = patients.withColumn(
    "lat",
    F.when(F.col("lat") < 5, F.col("lat") * 100)
     .otherwise(F.col("lat"))
)

patients.write.mode("overwrite").format("delta").saveAsTable("silver.patients")


## transforms medications

In [0]:
medications = spark.table("bronze.medications")
medications = clean_cols(medications)

medications = medications \
.withColumn("start", safe_timestamp("start")) \
.withColumn("stop", safe_timestamp("stop")) \
.withColumn("totalcost", F.col("totalcost").cast("double")) \
.withColumn("base_cost", F.col("base_cost").cast("double")) \
.dropDuplicates()

# Remove Extreme Outlier Cost
medications = medications.withColumn(
    "totalcost",
    F.when(F.col("totalcost") > 100000, None)
     .otherwise(F.col("totalcost"))
)

medications.write.mode("overwrite").format("delta").saveAsTable("silver.medications")


## transform conditions

In [0]:
conditions = spark.table("bronze.conditions")
conditions = clean_cols(conditions)

conditions = conditions \
.withColumn("patient", F.lower(F.trim(F.col("patient")))) \
.withColumn("encounter", F.lower(F.trim(F.col("encounter")))) \
.withColumn("start", safe_date("start")) \
.withColumn("stop", safe_date("stop")) \
.dropDuplicates()

conditions.write.mode("overwrite").format("delta").saveAsTable("silver.conditions")


## transformer symptoms

In [0]:
symptoms = spark.table("bronze.symptoms")
symptoms = clean_cols(symptoms)

symptoms = symptoms \
.withColumn("patient", F.lower(F.trim(F.col("patient")))) \
.dropDuplicates()

symptoms.write.mode("overwrite").format("delta").saveAsTable("silver.symptoms")


## transform encounters

In [0]:
encounters = spark.table("bronze.encounters")
encounters = clean_cols(encounters)

if "start" in encounters.columns:
    encounters = encounters.withColumn("start", safe_timestamp("start"))

if "stop" in encounters.columns:
    encounters = encounters.withColumn("stop", safe_timestamp("stop"))

if "patient" in encounters.columns:
    encounters = encounters.withColumn(
        "patient",
        F.lower(F.trim(F.col("patient")))
    )

encounters = encounters.dropDuplicates()

encounters.write.mode("overwrite").format("delta").saveAsTable("silver.encounters")


print("Silver Layer Complete")

Silver Layer Complete
